In [1]:
from shutil import rmtree
from pathlib import Path

import os
import gzip
import numpy as np
import matplotlib.pyplot as plt

from vot_utils.data import DATA_DIRECTORY

In [2]:
import torch

from torch import is_tensor
from torch import tensor
from torch.nn import Module
from torch.utils.data import Dataset, DataLoader

from adam_atan2_pytorch import MuonAdamAtan2

from einops import rearrange

import torchvision
import torchvision.transforms as T
from torchvision.utils import save_image

# from transfusion_pytorch import Transfusion, print_modality_sample
# from accelerate import Accelerator


In [4]:
# Prepare and clean data
# import os
# import gzip
# import re

# def clean_wiki_xml(input_stream, output_stream):
#     # Pre-compile regular expressions for performance
#     re_xml_tags = re.compile(r'<.*>')
#     re_ref = re.compile(r'<ref[^<]*</ref>')
#     re_xhtml = re.compile(r'<[^>]*>')
#     re_http = re.compile(r'\[http:[^\] ]*')
#     re_thumb = re.compile(r'\|thumb', re.IGNORECASE)
#     re_left = re.compile(r'\|left', re.IGNORECASE)
#     re_right = re.compile(r'\|right', re.IGNORECASE)
#     re_px = re.compile(r'\|\d+px', re.IGNORECASE)
#     re_image = re.compile(r'\[\[image:[^\[\]]*\|', re.IGNORECASE)
#     re_category = re.compile(r'\[\[category:([^|\]]*)[^\]]*\]\]', re.IGNORECASE)
#     re_interwiki = re.compile(r'\[\[[a-z\-]*:[^\]]*\]\]')
#     re_wiki_url = re.compile(r'\[\[[^\|\]]*\|')
#     re_icons = re.compile(r'{{[^}]*}}')
#     re_tables = re.compile(r'{[^}]*}')
#     re_url_enc = re.compile(r'&[^;]*;')
#     re_non_alpha = re.compile(r'[^a-zA-Z.,!?;:]+')

#     digit_map = {
#         '0': ' zero ', '1': ' one ', '2': ' two ', '3': ' three ',
#         '4': ' four ', '5': ' five ', '6': ' six ', '7': ' seven ',
#         '8': ' eight ', '9': ' nine '
#     }

#     full_text = input_stream.read()
#     records = [chunk + '>' for chunk in full_text.split('>')]
    
#     if records and not full_text.endswith('>'):
#         records[-1] = records[-1][:-1]

#     is_text = False

#     for record in records:
#         if '<text ' in record:
#             is_text = True
#         if re.search(r'#redirect', record, re.IGNORECASE):
#             is_text = False
            
#         if is_text:
#             if '</text>' in record:
#                 is_text = False
                
#             record = re_xml_tags.sub('', record)
#             record = record.replace('&amp;', '&')
#             record = record.replace('&lt;', '<')
#             record = record.replace('&gt;', '>')
#             record = re_ref.sub('', record)
#             record = re_xhtml.sub('', record)
#             record = re_http.sub('[', record)
#             record = re_thumb.sub('', record)
#             record = re_left.sub('', record)
#             record = re_right.sub('', record)
#             record = re_px.sub('', record)
#             record = re_image.sub('', record)
#             record = re_category.sub(r'[[\1]]', record)
#             record = re_interwiki.sub('', record)
#             record = re_wiki_url.sub('[[', record)
#             record = re_icons.sub('', record)
#             record = re_tables.sub('', record)
#             record = record.replace('[', '').replace(']', '')
#             record = re_url_enc.sub(' ', record)

#             record = f" {record} "
#             # record = record.lower()
                
#             record = re_non_alpha.sub(' ', record)

#             if record:
#                 record = record[:-1]
#                 output_stream.write(record)

# # --- NEW FILE HANDLING LOGIC ---
# if __name__ == '__main__':
#     # Define your directory (ensure this matches your actual setup)
    
#     # Define input and output paths
#     input_filepath = os.path.join(DATA_DIRECTORY, "enwik8", "enwik8.gz")
#     output_filepath = os.path.join(DATA_DIRECTORY, "enwik8", "enwik8_clean")
    
#     print(f"Reading and cleaning {input_filepath}...")
#     print("This might take a moment depending on your machine...")

#     # Open the gzip file in read-text ('rt') mode so it handles decoding automatically
#     with gzip.open(input_filepath, 'rt', encoding='utf-8', errors='ignore') as in_file:
#         # Open a standard text file to write the cleaned output
#         with open(output_filepath, 'w', encoding='utf-8') as out_file:
#             clean_wiki_xml(in_file, out_file)
            
#     print(f"Success! Cleaned file saved to {output_filepath}")

In [5]:
# constants

NUM_BATCHES = int(1e5)
BATCH_SIZE = 4
GRAD_ACCUM_EVERY = 4
LEARNING_RATE = 1e-4
VALIDATE_EVERY = 100
PRIME_LENGTH = 64
GENERATE_EVERY = 500
GENERATE_LENGTH = 256
SEQ_LEN = 256


In [6]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

In [7]:
# functions

def divisible_by(num, den):
    return (num % den) == 0


def cycle(loader):
    while True:
        for data in loader:
            yield data

In [8]:
with gzip.open(os.path.join(DATA_DIRECTORY, "enwik8", "enwik8_clean.gz")) as file:
    data = np.frombuffer(file.read(int(95e6)), dtype = np.uint8).copy()
    np_train, np_valid = np.split(data, [int(90e6)])
    data_train, data_val = torch.from_numpy(np_train), torch.from_numpy(np_valid)

In [9]:
class TextSamplerDataset(Dataset):
    def __init__(self, data, seq_len):
        super().__init__()
        self.data = data
        self.seq_len = seq_len
        self.data_length = data.shape[0]

    def __len__(self):
        return self.data.size(0) // self.seq_len

    def __getitem__(self, index):
        rand_start = torch.randint(0, self.data_length - self.seq_len, (1,))
        full_seq = self.data[rand_start : rand_start + self.seq_len + 1].long()
        return full_seq
    

In [10]:
from torch.optim import Adam


train_dataset = TextSamplerDataset(data_train, SEQ_LEN)
val_dataset = TextSamplerDataset(data_val, SEQ_LEN)
train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE)

# optimizer

# optim = Adam(model.parameters(), lr = LEARNING_RATE)

train_loader = cycle(train_loader)
val_loader = cycle(val_loader)

#### Manual Run

In [74]:
from functools import partial

from references.transfusion_pytorch.transfusion_pytorch.transfusion import (
    Transformer,
    default,
    exists,
    cast_tuple,
    default_to_modality_shape_fn,
    identity,
    add_temp_batch_dim,
    char_tokenize,
    decode_chars,
    append_dims,
    ModalityInfo,
    get_model_output_to_flow_fn,
    pack_one_with_inverse,
    max_neg_value
)

import torch.nn.functional as F

from torch import nn
from torch.nn import ModuleList
from torch.nn import Linear

from axial_positional_embedding import ContinuousAxialPositionalEmbedding
from rotary_embedding_torch import RotaryEmbedding, apply_rotary_emb
from torchdiffeq import odeint
from ema_pytorch import EMA

from beartype import beartype
from beartype.door import is_bearable

In [43]:
batch = next(train_loader)

In [44]:
num_text_tokens = 256
transformer = dict(dim=384, depth=8, dim_head=64, heads=8, attn_laser=True)

# All else is default
dim_latent = None
channel_first_latent = False
modality_default_shape = None
modality_encoder = None
modality_decoder = None
add_pos_emb = False
modality_num_dim = None

velocity_consistency_loss_weight = 0.1
reconstruction_loss_weight = 0.
model_output_clean = True

to_modality_shape_fn = default_to_modality_shape_fn
fallback_to_default_shape_if_invalid = False
modality_encoder_decoder_requires_batch_dim = True
pre_post_transformer_enc_dec = None

ignore_index = -1
flow_loss_weight = 1.0
text_loss_weight = 1.0

odeint_kwargs = dict(atol=1e-5, rtol=1e-5, method="midpoint")
eps = 1e-2
prob_uncond = 0.1

In [45]:
if isinstance(transformer, dict):
    transformer = Transformer(**transformer).to(device)

dim = transformer.dim
dim_latent = default(dim_latent, dim)

In [46]:
dim_latents = cast_tuple(dim_latent)
num_modalities = len(dim_latents)
channel_first_latent = cast_tuple(channel_first_latent, num_modalities)
to_modality_shape_fn = cast_tuple(to_modality_shape_fn, num_modalities)

if not exists(modality_default_shape) or is_bearable(modality_default_shape, tuple[int, ...]):
    modality_default_shape = (modality_default_shape,) * num_modalities

modality_num_dim = cast_tuple(modality_num_dim, num_modalities)
add_pos_emb = cast_tuple(add_pos_emb, num_modalities)


In [47]:
pos_emb_mlp = ModuleList([])

for modality_add_pos_emb, modality_ndim in zip(add_pos_emb, modality_num_dim):
    if not modality_add_pos_emb:
        pos_emb_mlp.append(None)
        continue

    pos_generating_mlp = ContinuousAxialPositionalEmbedding(
        dim = dim,
        num_axial_dims = modality_ndim,
    )

    pos_emb_mlp.append(pos_generating_mlp)


modality_encoder = cast_tuple(modality_encoder, 1 if exists(modality_encoder) else num_modalities)
modality_decoder = cast_tuple(modality_decoder, 1 if exists(modality_decoder) else num_modalities)

modality_encoder = ModuleList(modality_encoder)
modality_decoder = ModuleList(modality_decoder)

maybe_add_temp_batch_dim = add_temp_batch_dim if modality_encoder_decoder_requires_batch_dim else identity

In [59]:
num_text_special_ids = 3
sos_id, eos_id, null_text_id = (
    num_text_tokens,
    (num_text_tokens + 1),
    (num_text_tokens + 2),
)

num_modality_special_ids = num_modalities * 2
som_eom_tensor = torch.arange(num_modality_special_ids) + num_text_tokens + num_text_special_ids
som_eom_tensor = rearrange(som_eom_tensor, '(start_end m) -> start_end m', start_end = 2)
som_ids, eom_ids = som_eom_tensor.tolist()

meta_token_offset = num_text_tokens + num_text_special_ids + num_modality_special_ids
meta_id = meta_token_offset
num_meta_tokens = 128 + 1

char_tokenizer = partial(char_tokenize, offset = meta_token_offset + 1)
decode_chars = partial(decode_chars, offset = meta_token_offset + 1)


In [60]:
pre_post_transformer_enc_dec = cast_tuple(pre_post_transformer_enc_dec, num_modalities)

In [61]:
latent_to_model_projs = []
model_to_latent_projs = []

for (
    dim_latent,
    one_channel_first_latent,
    enc_dec,
) in zip(dim_latents, channel_first_latent, pre_post_transformer_enc_dec):
    pre_attend_enc, post_attend_dec = default(enc_dec, (None, None))

    latent_to_model_proj = Linear(dim_latent, dim) if dim_latent != dim else nn.Identity()
    model_to_latent_proj = Linear(dim, dim_latent, bias = False)

    latent_to_model_projs.append(default(pre_attend_enc, latent_to_model_proj))
    model_to_latent_projs.append(default(post_attend_dec, model_to_latent_proj))

latent_to_model_projs = ModuleList(latent_to_model_projs)
model_to_latent_projs = ModuleList(model_to_latent_projs)


In [78]:
rotary_emb = RotaryEmbedding(transformer.dim_head).to(device)

effective_num_text_tokens = num_text_tokens + num_text_special_ids + num_modality_special_ids + num_meta_tokens

text_embed = nn.Embedding(effective_num_text_tokens, dim).to(device)
to_text_logits = Linear(dim, effective_num_text_tokens, bias = False).to(device)
text_only_mask = (torch.arange(effective_num_text_tokens) < num_text_tokens).to(device)

In [63]:
has_recon_loss = reconstruction_loss_weight > 0.
odeint_fn = partial(odeint, **odeint_kwargs)

#### Run Forward Pass (Modality Only)

In [92]:
batch = next(train_loader)
# loss = model(batch, velocity_consistency_ema_model = ema_model)


In [93]:
modalities = batch
velocity_consistency_ema_model = None
decoding_text_or_modality = None
return_embed = False
return_loss = True
modality_type = None
times = None
cache = None
return_hiddens = False
return_kv_cache = False

In [94]:
is_decoding = exists(decoding_text_or_modality)
is_text_only = is_tensor(modalities) and modalities.dtype in (torch.int, torch.long)
is_modality_only = is_tensor(modalities) and modalities.dtype == torch.float

In [99]:
text = modalities
text = text.to(device)

if return_loss:
    text, labels = text[:, :-1], text[:, 1:]

# embed text

text = text.masked_fill(text == -1, 0)
tokens = text_embed(text)

# rotary

seq_len = tokens.shape[-2]
pos = torch.arange(seq_len, device = device)

rotary_emb_ = rotary_emb(pos)

# attention

transformer_out = transformer(
    tokens,
    rotary_emb = rotary_emb_,
    causal_mask = True,
    cache = cache,
    return_kv_cache = return_kv_cache,
    return_hiddens = True
)

embed, hiddens, *maybe_kv_cache = transformer_out
kv_cache = maybe_kv_cache[0] if return_kv_cache else None

In [116]:
text

tensor([[105, 102, 101,  ..., 117, 102,  97],
        [105, 111, 110,  ..., 101, 115, 115],
        [101, 114, 114,  ...,  97, 112, 101],
        [109,  32,  79,  ..., 101, 110, 100]], device='mps:0')

In [114]:
text_embed(text)

tensor([[[-7.8781e-01,  1.1446e+00,  3.5231e-01,  ..., -2.0381e+00,
           2.0197e-01,  6.5730e-01],
         [-3.4545e-01,  2.5505e-01,  6.0383e-01,  ..., -1.3312e+00,
           5.4447e-01,  8.6575e-01],
         [ 6.0701e-01,  1.4402e+00, -9.7017e-01,  ...,  3.0008e+00,
          -1.2203e+00, -1.8377e-02],
         ...,
         [ 1.0133e-01, -8.6487e-01,  3.4377e-01,  ...,  2.7425e-02,
           9.3442e-01, -1.3753e+00],
         [-3.4545e-01,  2.5505e-01,  6.0383e-01,  ..., -1.3312e+00,
           5.4447e-01,  8.6575e-01],
         [-2.1165e+00, -3.7989e-01, -1.5179e+00,  ..., -1.4919e+00,
          -3.4725e-01,  4.2523e-01]],

        [[-7.8781e-01,  1.1446e+00,  3.5231e-01,  ..., -2.0381e+00,
           2.0197e-01,  6.5730e-01],
         [ 8.3049e-01,  6.2632e-01, -6.2966e-01,  ..., -1.2849e-01,
           4.5142e-01,  1.2428e+00],
         [ 5.5946e-01, -7.8909e-01,  1.1426e+00,  ..., -1.2190e-01,
          -5.8315e-01,  3.2235e+00],
         ...,
         [ 6.0701e-01,  1

In [101]:
# Text unembedding

logits = to_text_logits(embed)

In [102]:
logits = logits.masked_fill(~text_only_mask, max_neg_value(logits))

In [103]:
logits.shape

torch.Size([4, 256, 390])

In [105]:
loss = F.cross_entropy(
    rearrange(logits, 'b n l -> b l n'),
    labels,
    ignore_index = ignore_index
)

In [109]:
rearrange(logits, 'b n l -> b l n').shape

torch.Size([4, 390, 256])

In [107]:
labels

tensor([[102, 101,  32,  ..., 102,  97,  99],
        [111, 110,  32,  ..., 115, 115,  32],
        [114, 114, 101,  ..., 112, 101,  32],
        [ 32,  79, 110,  ..., 110, 100, 101]], device='mps:0')